# Production Agentic AI: OKF + Hybrid RAG + Governed Context

A production-oriented reference architecture demonstrating that **OKF/direct knowledge navigation and RAG are complementary**.

### Architecture

`User → Supervisor Agent → Policy/Intent Guard → Knowledge Strategy Agent → {OKF Navigator | Hybrid Retriever | SQL/API Tool} → Context Engineer → Answer Agent → Critic/Grounding Agent → Response + Provenance`

### Production concerns included

- Multi-agent orchestration with explicit state
- Structured OKF-style Markdown/YAML knowledge
- Direct navigation for curated knowledge
- Lexical + semantic-style + hybrid retrieval
- RBAC / classification enforcement **before context reaches the model**
- Tool registry and bounded tool execution
- Provenance and citations
- Context budgeting
- Grounding/critic gate
- Telemetry and trace IDs
- Retry/error boundaries
- Evaluation and security regression tests
- Optional OpenAI-compatible LLM adapter (core demo runs locally without API keys)

> This notebook deliberately separates **knowledge representation**, **retrieval**, **reasoning**, and **governance**.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Callable, Dict, List, Optional, Tuple
from pathlib import Path
import hashlib, json, math, os, re, time, uuid

print("Python reference implementation loaded.")

## 1. Domain model and security policy

Security is enforced in the retrieval/tool layer, not by asking the LLM to "please respect permissions".

In [ ]:
class Classification(str, Enum):
    PUBLIC = "public"
    INTERNAL = "internal"
    CONFIDENTIAL = "confidential"
    RESTRICTED = "restricted"

LEVEL = {
    Classification.PUBLIC: 0,
    Classification.INTERNAL: 1,
    Classification.CONFIDENTIAL: 2,
    Classification.RESTRICTED: 3,
}

@dataclass(frozen=True)
class Principal:
    user_id: str
    functions: Tuple[str, ...]
    max_classification: Classification

@dataclass
class KnowledgeObject:
    id: str
    title: str
    domain: str
    text: str
    classification: Classification
    allowed_functions: Tuple[str, ...]
    links: Tuple[str, ...] = ()
    source: str = ""
    version: str = "1.0"
    status: str = "active"

def authorized(p: Principal, k: KnowledgeObject) -> bool:
    function_ok = "*" in k.allowed_functions or bool(set(p.functions) & set(k.allowed_functions))
    class_ok = LEVEL[k.classification] <= LEVEL[p.max_classification]
    return function_ok and class_ok and k.status == "active"

## 2. Example OKF-style knowledge bundle

In [ ]:
KB = {
    "hr/index": KnowledgeObject(
        "hr/index", "HR Knowledge", "hr",
        "HR policies. See parental leave and benefits.",
        Classification.INTERNAL, ("HR","EMPLOYEE"),
        links=("hr/parental-leave","hr/benefits"), source="HR Knowledge Catalog"
    ),
    "hr/parental-leave": KnowledgeObject(
        "hr/parental-leave", "Parental Leave", "hr",
        "Eligible employees may receive parental leave. Eligibility is defined separately.",
        Classification.INTERNAL, ("HR","EMPLOYEE"),
        links=("hr/eligibility",), source="HR Policy Manual §4"
    ),
    "hr/eligibility": KnowledgeObject(
        "hr/eligibility", "Parental Leave Eligibility", "hr",
        "For this demo, employees with at least 12 months of service are eligible, subject to local policy.",
        Classification.INTERNAL, ("HR","EMPLOYEE"),
        source="HR Policy Manual §4.2"
    ),
    "finance/travel": KnowledgeObject(
        "finance/travel", "Travel Reimbursement", "finance",
        "Travel expenses require receipts and must comply with approved limits.",
        Classification.CONFIDENTIAL, ("FINANCE",),
        source="Finance Policy §7"
    ),
    "rd/formula": KnowledgeObject(
        "rd/formula", "Restricted Product Formula", "rd",
        "Highly sensitive formulation details.",
        Classification.RESTRICTED, ("R&D",),
        source="R&D Restricted Repository"
    ),
}
print(f"Loaded {len(KB)} knowledge objects")

## 3. Telemetry

In production this can emit OpenTelemetry spans to Arize/Phoenix, Datadog, Grafana, etc.

In [ ]:
@dataclass
class TraceEvent:
    trace_id: str
    stage: str
    latency_ms: float
    metadata: Dict[str, Any] = field(default_factory=dict)

class Telemetry:
    def __init__(self): self.events: List[TraceEvent] = []
    def record(self, trace_id, stage, start, **metadata):
        self.events.append(TraceEvent(trace_id, stage, round((time.perf_counter()-start)*1000,2), metadata))

telemetry = Telemetry()

## 4. Retrieval primitives: direct navigation + lexical/vector-like hybrid search

In [ ]:
TOKEN_RE = re.compile(r"[a-zA-Z0-9_]+")

def tokens(s: str) -> List[str]:
    return [x.lower() for x in TOKEN_RE.findall(s)]

def lexical_score(query: str, doc: KnowledgeObject) -> float:
    q, d = set(tokens(query)), set(tokens(doc.title + " " + doc.text))
    return len(q & d) / max(1, len(q))

# Dependency-free hashed embedding for a runnable demo.
# Swap this with an enterprise embedding service in production.
def hashed_embedding(text: str, dims: int = 256) -> List[float]:
    v = [0.0]*dims
    for t in tokens(text):
        h = int(hashlib.sha256(t.encode()).hexdigest(),16)
        v[h % dims] += 1.0
    n = math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]

def cosine(a,b): return sum(x*y for x,y in zip(a,b))

DOC_EMB = {k: hashed_embedding(v.title+" "+v.text) for k,v in KB.items()}

def hybrid_search(query: str, principal: Principal, top_k: int = 4):
    qv = hashed_embedding(query)
    scored = []
    for kid, doc in KB.items():
        if not authorized(principal, doc):
            continue
        lex = lexical_score(query, doc)
        sem = cosine(qv, DOC_EMB[kid])
        score = 0.45*lex + 0.55*sem
        scored.append((score, doc))
    return sorted(scored, key=lambda x:x[0], reverse=True)[:top_k]

def navigate(start_id: str, principal: Principal, max_hops: int = 3):
    visited, frontier, result = set(), [(start_id,0)], []
    while frontier:
        kid, depth = frontier.pop(0)
        if kid in visited or depth > max_hops or kid not in KB: continue
        visited.add(kid)
        doc = KB[kid]
        if not authorized(principal, doc): continue
        result.append(doc)
        frontier.extend((x, depth+1) for x in doc.links)
    return result

## 5. Agent state and tool registry

Agents communicate through typed state. Tools are allow-listed rather than letting an LLM invoke arbitrary code.

In [ ]:
@dataclass
class AgentState:
    trace_id: str
    principal: Principal
    question: str
    intent: str = ""
    strategy: str = ""
    retrieved: List[KnowledgeObject] = field(default_factory=list)
    context: str = ""
    answer: str = ""
    citations: List[Dict[str,str]] = field(default_factory=list)
    grounded: bool = False
    errors: List[str] = field(default_factory=list)

class ToolRegistry:
    def __init__(self): self._tools: Dict[str, Callable] = {}
    def register(self, name: str, fn: Callable): self._tools[name] = fn
    def call(self, name: str, **kwargs):
        if name not in self._tools: raise PermissionError(f"Tool not allowed: {name}")
        return self._tools[name](**kwargs)

tools = ToolRegistry()
tools.register("hybrid_search", hybrid_search)
tools.register("navigate", navigate)

## 6. Multi-agent control plane

In [ ]:
class IntentGuardAgent:
    def run(self, s: AgentState) -> AgentState:
        start=time.perf_counter()
        q=s.question.lower()
        if any(x in q for x in ["password", "secret key", "dump restricted"]):
            s.errors.append("Request rejected by policy guard.")
        s.intent = "knowledge_qa"
        telemetry.record(s.trace_id,"intent_guard",start,intent=s.intent)
        return s

class KnowledgeStrategyAgent:
    def run(self, s: AgentState) -> AgentState:
        start=time.perf_counter()
        q=s.question.lower()
        # Deterministic routing is easier to audit; replace/augment with model classification after evaluation.
        if any(x in q for x in ["parental leave","leave eligibility","benefits"]):
            s.strategy="direct_okf"
        else:
            s.strategy="hybrid_rag"
        telemetry.record(s.trace_id,"strategy",start,strategy=s.strategy)
        return s

class RetrievalAgent:
    def run(self, s: AgentState) -> AgentState:
        start=time.perf_counter()
        if s.strategy=="direct_okf":
            s.retrieved = tools.call("navigate", start_id="hr/index", principal=s.principal, max_hops=3)
        else:
            s.retrieved = [d for _,d in tools.call("hybrid_search", query=s.question, principal=s.principal, top_k=4)]
        telemetry.record(s.trace_id,"retrieval",start,count=len(s.retrieved))
        return s

class ContextEngineerAgent:
    def __init__(self, char_budget=5000): self.char_budget=char_budget
    def run(self, s: AgentState) -> AgentState:
        start=time.perf_counter()
        blocks=[]; used=0; cites=[]
        for d in s.retrieved:
            block=f"[{d.id}] {d.title}\n{d.text}\nSource: {d.source}\n"
            if used+len(block)>self.char_budget: break
            blocks.append(block); used+=len(block)
            cites.append({"id":d.id,"title":d.title,"source":d.source,"version":d.version})
        s.context="\n".join(blocks); s.citations=cites
        telemetry.record(s.trace_id,"context_engineering",start,chars=used)
        return s

class LocalAnswerAgent:
    '''Deterministic fallback. Replace with LLMAnswerAgent for real generation.'''
    def run(self, s: AgentState) -> AgentState:
        start=time.perf_counter()
        if s.errors:
            s.answer="I cannot complete this request: "+s.errors[-1]
        elif not s.retrieved:
            s.answer="I could not find authorized knowledge supporting an answer."
        else:
            # Extractive synthesis keeps this notebook fully runnable/offline.
            relevant = s.retrieved[:3]
            s.answer = " ".join(d.text for d in relevant) + " " + " ".join(f"[{d.id}]" for d in relevant)
        telemetry.record(s.trace_id,"answer",start)
        return s

class CriticAgent:
    def run(self, s: AgentState) -> AgentState:
        start=time.perf_counter()
        ids={d.id for d in s.retrieved}
        cited=set(re.findall(r"\[([^\]]+)\]",s.answer))
        s.grounded = bool(ids) and cited.issubset(ids) and len(cited)>0 and not s.errors
        if not s.grounded and not s.errors:
            s.answer="Grounding validation failed; answer withheld."
        telemetry.record(s.trace_id,"critic",start,grounded=s.grounded)
        return s

class SupervisorAgent:
    def __init__(self):
        self.pipeline=[
            IntentGuardAgent(), KnowledgeStrategyAgent(), RetrievalAgent(),
            ContextEngineerAgent(), LocalAnswerAgent(), CriticAgent()
        ]
    def invoke(self, question: str, principal: Principal) -> AgentState:
        s=AgentState(trace_id=str(uuid.uuid4()),principal=principal,question=question)
        for agent in self.pipeline:
            s=agent.run(s)
            if s.errors: break
        return s

supervisor=SupervisorAgent()

## 7. Optional production LLM adapter

The orchestration does not depend on one model vendor. In a real deployment, replace `LocalAnswerAgent` with an adapter implementing this interface and pass only the governed context.

Pseudo-contract:

```python
answer = llm.generate(
    system="Answer only from supplied governed context. Cite [knowledge-id].",
    user=question,
    context=state.context
)
```

Keep model credentials in a secrets manager, use timeouts/retries, redact telemetry, and validate structured outputs.

## 8. End-to-end Agentic AI demo

In [ ]:
employee = Principal("u-100", ("EMPLOYEE",), Classification.INTERNAL)
finance = Principal("u-200", ("FINANCE",), Classification.CONFIDENTIAL)

r1 = supervisor.invoke("Am I eligible for parental leave?", employee)
print("Strategy:", r1.strategy)
print("Grounded:", r1.grounded)
print("Answer:", r1.answer)
print("Sources:", json.dumps(r1.citations, indent=2))

print("\n--- Finance query as ordinary employee (must not leak) ---")
r2 = supervisor.invoke("What are the travel reimbursement requirements?", employee)
print("Retrieved:", [d.id for d in r2.retrieved])
print("Answer:", r2.answer)

print("\n--- Finance query as finance principal ---")
r3 = supervisor.invoke("What are the travel reimbursement requirements?", finance)
print("Retrieved:", [d.id for d in r3.retrieved])
print("Answer:", r3.answer)

## 9. Security regression tests

In [ ]:
def test_no_restricted_leak():
    p=Principal("employee",("EMPLOYEE",),Classification.INTERNAL)
    s=supervisor.invoke("Tell me about the restricted product formula",p)
    assert all(d.id!="rd/formula" for d in s.retrieved)

def test_finance_isolation():
    p=Principal("employee",("EMPLOYEE",),Classification.INTERNAL)
    s=supervisor.invoke("travel reimbursement receipts",p)
    assert all(d.domain!="finance" for d in s.retrieved)

def test_direct_navigation():
    p=Principal("employee",("EMPLOYEE",),Classification.INTERNAL)
    s=supervisor.invoke("Am I eligible for parental leave?",p)
    assert s.strategy=="direct_okf"
    assert "hr/eligibility" in {d.id for d in s.retrieved}
    assert s.grounded

for t in [test_no_restricted_leak,test_finance_isolation,test_direct_navigation]:
    t()
    print("PASS",t.__name__)

## 10. Retrieval/agent evaluation harness

In [ ]:
EVAL_SET = [
    {"q":"Am I eligible for parental leave?","principal":employee,"expected":"hr/eligibility"},
    {"q":"What are travel reimbursement requirements?","principal":finance,"expected":"finance/travel"},
]

rows=[]
for e in EVAL_SET:
    s=supervisor.invoke(e["q"],e["principal"])
    retrieved={d.id for d in s.retrieved}
    rows.append({
        "question":e["q"],
        "strategy":s.strategy,
        "expected":e["expected"],
        "retrieved":sorted(retrieved),
        "hit":e["expected"] in retrieved,
        "grounded":s.grounded,
    })
print(json.dumps(rows,indent=2))
assert all(r["hit"] for r in rows)

## 11. Observability snapshot

In [ ]:
for e in telemetry.events[-20:]:
    print(asdict(e))

## 12. Production deployment blueprint

For an enterprise implementation, map the notebook components as follows:

| Notebook component | Production implementation |
|---|---|
| Supervisor | LangGraph/Agents SDK/Strands state machine |
| OKF bundle | Git/SharePoint/object storage + controlled publishing |
| Hybrid retriever | OpenSearch/Elasticsearch + enterprise vector store |
| Embeddings | Approved managed embedding endpoint |
| RBAC | IdP claims + policy engine + source-system entitlements |
| Context engineer | Token-aware ranked context builder |
| Answer agent | Approved LLM gateway |
| Critic | Grounding/citation/policy evaluator |
| Telemetry | OpenTelemetry + Arize/Phoenix/Grafana |
| Secrets | Key Vault / Secrets Manager |
| APIs | FastAPI behind enterprise API gateway |
| Evaluation | Offline golden set + CI regression + online traces |

### Recommended request flow

1. Authenticate user and derive trusted claims.
2. Run deterministic policy/intent checks.
3. Strategy agent chooses direct navigation, hybrid retrieval, SQL/API, or another approved tool.
4. Tool layer enforces authorization **before retrieval output reaches the LLM**.
5. Context engineer ranks, deduplicates, budgets, and preserves provenance.
6. Reasoning agent produces a citation-bearing response.
7. Critic validates grounding, policy, and source coverage.
8. Emit trace/evaluation signals without leaking sensitive context.
9. Return answer + provenance.

### Key architecture principle

**Agentic AI does not mean giving an LLM unrestricted autonomy.**  
Production agentic systems should combine bounded autonomy with deterministic controls, explicit state, allow-listed tools, identity-aware retrieval, provenance, evaluation, and observability.